## Multi-Agent System (MAS)

MAS란, 여러 개의 에이전트가 협력해서 하나의 목표를 달성하는 시스템입니다.

하나의 에이전트가 모든 일을 처리하는 대신, 각자 역할이 다른 여러 에이전트가 분업해서 처리합니다.

```
사용자 질문
     ↓
Supervisor Agent  ← 질문을 보고 알맞은 sub-agent에게 전달
     ├── 임직원 Agent  : 직원 정보 조회
     ├── 제품 Agent    : 제품 카탈로그 + 회사소개 검색
     └── 규정 Agent    : 사내 규정 검색
```

핵심 아이디어는 **sub-agent를 tool로 등록**하는 것입니다.
supervisor는 어떤 sub-agent를 부를지만 결정하고, 실제 처리는 sub-agent가 합니다.

### 환경 설정

In [ ]:
import os
import uuid
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

from rag_retriever import RAGRetriever

load_dotenv(find_dotenv())

credential_key = os.getenv("credential_key")
send_system_name = os.getenv("send_system_name")
model_name = os.getenv("model")
api_base_url = os.getenv("api_base_url")
user_id = os.getenv("user_id")

os.environ["OPENAI_API_KEY"] = 'api_key'

llm = ChatOpenAI(
    model=model_name,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticket': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
)

### 1. Sub-agent 준비하기

각 sub-agent는 자신의 역할에 맞는 tool과 system prompt를 가집니다.

- **임직원 Agent** : 엑셀 파일을 pandas로 직접 조회 (이름으로 정확하게 검색)
- **제품 Agent** : 제품 카탈로그(PDF)와 회사소개(PPT)를 RAG로 검색
- **규정 Agent** : 사내 규정 문서(Word)를 RAG로 검색

#### 1-1. 임직원 Agent

In [ ]:
@tool
def search_employee(name: str) -> str:
    """임직원 이름으로 정보를 검색합니다. 연락처, 부서, 담당업무 등을 확인할 수 있습니다."""
    df = pd.read_excel("../04_RAG/data/키키테크_임직원및프로젝트현황.xlsx", header=1)
    result = df[df["성명"] == name]

    if result.empty:
        return f"'{name}' 이름의 임직원을 찾을 수 없습니다."

    row = result.iloc[0]
    return "\n".join(f"{col}: {row[col]}" for col in df.columns)


employee_agent = create_agent(
    llm,
    tools=[search_employee],
    system_prompt="당신은 키키테크 임직원 정보 전문 에이전트입니다. 임직원의 이름을 파악해서 search_employee 도구로 정보를 찾아 답변하세요."
)

#### 1-2. 제품 Agent

In [ ]:
# 제품 관련 문서로 RAGRetriever 구성
product_rag = RAGRetriever()
product_rag.add_file("../04_RAG/data/키키테크_AI솔루션_제품카탈로그.pdf")
product_rag.add_file("../04_RAG/data/키키테크_회사소개.pptx")
product_rag.build_retriever()

@tool
def search_product(query: str) -> str:
    """키키테크 제품 정보와 회사 소개를 검색합니다."""
    results = product_rag.search(query)
    return "\n\n---\n\n".join(doc.page_content for doc in results)


product_agent = create_agent(
    llm,
    tools=[search_product],
    system_prompt="당신은 키키테크 제품 전문 에이전트입니다. search_product 도구로 제품 정보를 찾아 답변하세요."
)

#### 1-3. 규정 Agent

In [ ]:
# 사내 규정 문서로 RAGRetriever 구성
policy_rag = RAGRetriever()
policy_rag.add_file("../04_RAG/data/키키테크_사내규정_행동강령.docx")
policy_rag.build_retriever()

@tool
def search_policy(query: str) -> str:
    """키키테크 사내 규정과 행동강령을 검색합니다."""
    results = policy_rag.search(query)
    return "\n\n---\n\n".join(doc.page_content for doc in results)


policy_agent = create_agent(
    llm,
    tools=[search_policy],
    system_prompt="당신은 키키테크 사내 규정 전문 에이전트입니다. search_policy 도구로 규정을 찾아 답변하세요."
)

### 2. Sub-agent를 tool로 감싸기

supervisor가 sub-agent를 호출하려면 sub-agent를 **tool로 감싸야** 합니다.

tool의 docstring을 보고 supervisor가 어느 sub-agent를 부를지 결정하기 때문에,
각 tool의 역할 설명을 명확하게 작성하는 것이 중요합니다.

In [ ]:
@tool
def ask_employee_agent(query: str) -> str:
    """임직원 이름, 연락처, 부서, 담당업무, 프로젝트 현황 등 직원 관련 질문을 처리합니다."""
    result = employee_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content


@tool
def ask_product_agent(query: str) -> str:
    """키키테크 제품 소개, 기능, 가격, 회사 개요 등 제품과 회사 관련 질문을 처리합니다."""
    result = product_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content


@tool
def ask_policy_agent(query: str) -> str:
    """근무 규정, 복리후생, 행동강령, 보안 정책 등 사내 규정 관련 질문을 처리합니다."""
    result = policy_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

### 3. Supervisor Agent 만들기

supervisor는 사용자의 질문을 보고 알맞은 sub-agent tool을 선택합니다.

직접 답을 생성하지 않고, **라우팅과 결과 취합**만 담당합니다.

In [ ]:
supervisor = create_agent(
    llm,
    tools=[ask_employee_agent, ask_product_agent, ask_policy_agent],
    system_prompt=(
        "당신은 키키테크 사내 Q&A 코디네이터입니다. "
        "질문의 내용을 파악해서 알맞은 전문 에이전트에게 전달하고, 받은 답변을 사용자에게 전달하세요. "
        "키키테크와 무관한 질문에는 답하지 마세요."
    )
)

### 4. 실행해보기

In [ ]:
# 임직원 관련 질문 → 임직원 Agent로 라우팅
response = supervisor.invoke(
    {"messages": [{"role": "user", "content": "이서연 책임의 내선번호가 뭐야?"}]}
)
print(response["messages"][-1].content)

In [ ]:
# 제품 관련 질문 → 제품 Agent로 라우팅
response = supervisor.invoke(
    {"messages": [{"role": "user", "content": "TalkBridge Enterprise 요금제가 어떻게 돼?"}]}
)
print(response["messages"][-1].content)

In [ ]:
# 규정 관련 질문 → 규정 Agent로 라우팅
response = supervisor.invoke(
    {"messages": [{"role": "user", "content": "재택근무는 일주일에 몇 번까지 가능해?"}]}
)
print(response["messages"][-1].content)

### 실습

오늘 배운 RAG와 MAS 구조를 참고해서 키키테크와 관련된 QA를 처리하는 멀티 에이전트 챗봇을 프로젝트 형태로 구현해 봅시다!

`가이드라인.md`를 참고해 보세요